# LADA on Google Colab — Notebook 2 / 2: EVAL + PREDICTIONS

Loads the per-step checkpoints saved by the **train notebook** (`lada_colab.ipynb`) and, for each step:
- runs LADA's default eval (**no selector**) → reconstructs the per-step accuracy matrix (**Table 7**) and saves per-sample predictions,
- runs eval **with the zero-shot-CLIP selector** → saves predictions + task routing (**Figure 3**).

No retraining. Reuses the repo's own `Trainer.test_wo_selector` / `test_w_selector` so the numbers match training-time logs exactly.

**Run the train notebook first** (it downloads X-TAIL to Drive and writes the step checkpoints).
Set `Runtime > Change runtime type > GPU`.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU! Set Runtime > Change runtime type > GPU, then re-run.')

## 2. Config — must match the train notebook

In [ ]:
# These must match the values used in the TRAIN notebook (lada_colab.ipynb).
USE_DRIVE  = True
SHOTS      = 16
ORDER      = 'I'
RUN_NAME   = f'TAIL_{SHOTS}shot_order{ORDER}'
OUTPUT_DIR = 'TAIL_eval'   # local ./output dir for this eval session (scratch; harmless)
DUMMY_TASK = 'dtd'         # any small task; only used to build the model shell + the X-TAIL test loader
print('RUN_NAME =', RUN_NAME, '| USE_DRIVE =', USE_DRIVE)

## 3. Clone the repo and install dependencies

In [ ]:
%cd /content
![ -d lada ] || git clone https://github.com/maolinluo/lada.git
%cd /content/lada
!pip -q install -r requirements.txt modelscope
print('done')

## 4. Mount Drive, locate datasets + checkpoints, write the config
Datasets and step checkpoints come from the train notebook's Drive folder.

In [ ]:
import os, glob

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = '/content/drive/MyDrive/Project/LADA/X-TAIL'
else:
    DATASET_ROOT = '/content/datasets/X-TAIL'
assert os.path.isdir(DATASET_ROOT), (
    f'Dataset root not found: {DATASET_ROOT}. Run the TRAIN notebook first (it downloads X-TAIL).')

# Order-I full sequence (must match training).
seq_full = ['aircraft', 'caltech101', 'dtd', 'eurosat', 'flowers',
            'food101', 'mnist', 'oxford_pets', 'stanford_cars', 'sun397']
with open('configs/data/TAIL.yaml', 'w') as f:
    f.write('dataset_sequence: %s\n' % seq_full)
    f.write('root: "%s"\n' % DATASET_ROOT)
print(open('configs/data/TAIL.yaml').read())

RUN_BASE = (f'/content/drive/MyDrive/Project/LADA/runs/{RUN_NAME}' if USE_DRIVE
            else f'/content/lada/runs/{RUN_NAME}')
CKPT_DIR = os.path.join(RUN_BASE, 'checkpoints')
PRED_DIR = os.path.join(RUN_BASE, 'predictions')
os.makedirs(PRED_DIR, exist_ok=True)
assert os.path.isdir(CKPT_DIR), f'No checkpoints at {CKPT_DIR}. Run the TRAIN notebook first.'

ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'step*_*.pth.tar')))
print(f'\nFound {len(ckpts)} step checkpoints:')
for c in ckpts:
    print('  ', os.path.basename(c))

## 5. Build the eval model + helpers
We build a `Trainer` once (with a tiny dummy current task so it constructs the model shell and the
X-TAIL test loader over all 10 tasks). For each step we then overwrite the model's LADA features /
classifier / prototypes from that step's checkpoint — exactly what `test_wo_selector` consumes.

In [ ]:
import os, random
import numpy as np
import torch
from utils.config import _C as cfg
from trainer import Trainer

cfg.defrost()
cfg.merge_from_file('configs/data/TAIL.yaml')
cfg.merge_from_file('configs/model/clip_vit_b16.yaml')
cfg.merge_from_list(['num_shots', str(SHOTS), 'dataset', DUMMY_TASK,
                     'continue_train_first', 'True',           # skip auto-load; build LADA shell once
                     'num_workers', str(os.cpu_count() or 8),
                     'output_dir', OUTPUT_DIR])
cfg.output_dir = os.path.join('./output', cfg.output_dir)
if cfg.model_dir is None:
    cfg.model_dir = cfg.output_dir
os.makedirs(cfg.output_dir, exist_ok=True)

random.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0)

print('Building eval Trainer (dummy current task =', DUMMY_TASK, ') ...')
trainer = Trainer(cfg)                 # builds X-TAIL test loader (all 10 tasks) + model shell
device = trainer.device
NUM_CLASSES  = len(trainer.merged_classnames)
TASK_INDICES = list(trainer.indices)   # task boundaries in the merged label space
DATASETS     = list(cfg.dataset_sequence)
print('Merged classes:', NUM_CLASSES, '| task boundaries:', TASK_INDICES)

@torch.no_grad()
def load_step(ckpt_path):
    ck = torch.load(ckpt_path, map_location=device, weights_only=True)
    m = trainer.model
    m.dpt.image_prototypes         = ck['image_prototypes'].to(device)
    m.dpt.image_prototypes_covs    = ck['image_prototypes_covs'].to(device)
    m.dpt.image_prototypes_weights = ck['image_prototypes_weights'].to(device)
    m.dpt.text_prototypes          = ck['text_prototypes'].to(device)
    m.lada.prev_lada_features      = ck['lada_features']
    m.lada.joint_classifier        = ck['classifier']

def harvest():
    ev = trainer.evaluator
    return (np.array(ev._y_true), np.array(ev._y_pred),
            np.array(ev._task_true), np.array(ev._task_pred))

def per_task_acc(y_true, y_pred):
    correct = (y_true == y_pred).astype(float)
    bounds = list(TASK_INDICES) + [NUM_CLASSES]
    accs = []
    for i in range(len(TASK_INDICES)):
        lo, hi = bounds[i], bounds[i + 1]
        mask = (y_true >= lo) & (y_true < hi)
        accs.append(100.0 * correct[mask].mean() if mask.any() else float('nan'))
    return accs

def avg_task_recall(task_true, task_pred):
    ids = np.unique(task_true)
    recs = [(task_pred[task_true == i] == i).mean() for i in ids]
    return 100.0 * float(np.mean(recs))

## 6. Run eval over every step checkpoint
For each step: `test_wo_selector` (Table 7 row + Figure 2) then `test_w_selector` (Figure 3).
Per-sample predictions + task routing are saved to `predictions/stepNN_{wo,w}.npz` on Drive.

In [ ]:
import re

matrix = []        # rows = after training step k (no-selector); used for Table 7 / Figure 2
fig3_rows = []     # (step, dataset, acc_wo, acc_w, recall_wo, recall_w) for Figure 3

for c in ckpts:
    base = os.path.basename(c)
    mobj = re.match(r'step(\d+)_(.+)\.pth\.tar', base)
    k, ds = int(mobj.group(1)), mobj.group(2)
    load_step(c)

    # --- LADA default: no selector -> Table 7 row + Figure 2 curve ---
    trainer.test_wo_selector()
    yt, yp, tt, tp = harvest()
    np.savez(os.path.join(PRED_DIR, f'step{k:02d}_wo.npz'),
             y_true=yt, y_pred=yp, task_true=tt, task_pred=tp,
             indices=np.array(TASK_INDICES))
    matrix.append(per_task_acc(yt, yp))
    acc_wo = 100.0 * (yt == yp).mean(); rec_wo = avg_task_recall(tt, tp)

    # --- with zero-shot-CLIP selector -> Figure 3 comparison ---
    trainer.test_w_selector()
    yt2, yp2, tt2, tp2 = harvest()
    np.savez(os.path.join(PRED_DIR, f'step{k:02d}_w.npz'),
             y_true=yt2, y_pred=yp2, task_true=tt2, task_pred=tp2,
             indices=np.array(TASK_INDICES))
    acc_w = 100.0 * (yt2 == yp2).mean(); rec_w = avg_task_recall(tt2, tp2)

    fig3_rows.append((k, ds, acc_wo, acc_w, rec_wo, rec_w))
    print(f'step {k:02d} {ds:<14} acc(wo/w)={acc_wo:5.1f}/{acc_w:5.1f}'
          f'  taskRecall(wo/w)={rec_wo:5.1f}/{rec_w:5.1f}')

matrix = np.array(matrix)
np.save(os.path.join(RUN_BASE, 'matrix_wo.npy'), matrix)
print('\nSaved per-sample predictions to', PRED_DIR)

## 7. Reconstruct Table 7 (per-step matrix + Transfer / Average / Last)

In [ ]:
short = [d[:3] + '.' for d in DATASETS]
K = len(DATASETS)

print('=' * 78)
print('after\\on'.ljust(14) + ''.join(s.ljust(6) for s in short))
print('-' * 78)
for i, ds in enumerate(DATASETS):
    print(ds.ljust(14) + ''.join((f'{matrix[i][j]:.1f}').ljust(6) for j in range(K)))

transfer = [(np.mean([matrix[j][k] for j in range(k)]) if k > 0 else None) for k in range(K)]
average  = [np.mean([matrix[j][k] for j in range(K)]) for k in range(K)]
last     = [matrix[K - 1][k] for k in range(K)]

def fmt(v):
    return ('N/A'.ljust(6) if v is None else (f'{v:.1f}').ljust(6))

print('=' * 78)
print('Transfer'.ljust(14) + ''.join(fmt(v) for v in transfer))
print('Average'.ljust(14)  + ''.join(fmt(v) for v in average))
print('Last'.ljust(14)     + ''.join(fmt(v) for v in last))
print('=' * 78)
tmean = np.mean([v for v in transfer if v is not None])
print(f'Transfer Mean: {tmean:.1f}   (paper Table 1/7: 61.5)')
print(f'Average  Mean: {np.mean(average):.1f}   (paper Table 1/7: 72.7)')
print(f'Last     Mean: {np.mean(last):.1f}   (paper Table 1/7: 83.1)')

# Save a text copy next to the predictions.
with open(os.path.join(RUN_BASE, 'table7_reconstructed.txt'), 'w') as f:
    f.write('after/on'.ljust(14) + ''.join(s.ljust(6) for s in short) + '\n')
    for i, ds in enumerate(DATASETS):
        f.write(ds.ljust(14) + ''.join((f'{matrix[i][j]:.1f}').ljust(6) for j in range(K)) + '\n')
    f.write('Transfer'.ljust(14) + ''.join(fmt(v) for v in transfer) + '\n')
    f.write('Average'.ljust(14)  + ''.join(fmt(v) for v in average) + '\n')
    f.write('Last'.ljust(14)     + ''.join(fmt(v) for v in last) + '\n')
    f.write(f'Transfer Mean: {tmean:.1f}\nAverage Mean: {np.mean(average):.1f}\nLast Mean: {np.mean(last):.1f}\n')
print('Saved ->', os.path.join(RUN_BASE, 'table7_reconstructed.txt'))

## 8. Save Figure-3 data (selector vs no-selector)

In [ ]:
import csv
csv_path = os.path.join(RUN_BASE, 'fig3.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['step', 'dataset', 'acc_wo', 'acc_w', 'recall_wo', 'recall_w'])
    w.writerows(fig3_rows)
print('Saved Figure-3 data ->', csv_path)
for r in fig3_rows:
    print(r)

## 9. Plots (Figure 2 = LADA curves, Figure 3 = selector study)
LADA-only curves (the paper overlays ZSCL / MoE-Adapters / RAIL, which need their own runs).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Figure 2: per-dataset accuracy across learning steps (LADA only) ---
steps = np.arange(1, len(DATASETS) + 1)
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for j, ds in enumerate(DATASETS):
    ax = axes[j // 5][j % 5]
    ax.plot(steps, matrix[:, j], '-o', ms=3, label='LADA')
    ax.set_title(ds); ax.set_xlabel('step'); ax.set_ylim(0, 100)
axes[0][0].set_ylabel('accuracy (%)')
fig.suptitle('Figure 2 (LADA only): accuracy across learning steps — 16-shot Order-I')
fig.tight_layout()
fig.savefig(os.path.join(RUN_BASE, 'figure2_lada.png'), dpi=120)
plt.show()

# --- Figure 3: zero-shot-CLIP selector vs no selector ---
arr = np.array([(r[0], r[2], r[3], r[4], r[5]) for r in fig3_rows], dtype=float)  # step, acc_wo, acc_w, rec_wo, rec_w
k = arr[:, 0]
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.bar(k, arr[:, 1] - arr[:, 2], alpha=0.3, color='green', label='acc diff (wo - w)')
ax2.plot(k, arr[:, 3], '-o', color='C0', label='task recall w/o selector')
ax2.plot(k, arr[:, 4], '-s', color='C1', label='task recall w/ selector')
ax1.set_xlabel('learning step'); ax1.set_ylabel('accuracy difference (%)')
ax2.set_ylabel('task recall (%)')
ax1.set_title('Figure 3: zero-shot CLIP as selector vs not — 16-shot Order-I')
ax2.legend(loc='lower right')
fig.tight_layout()
fig.savefig(os.path.join(RUN_BASE, 'figure3_selector.png'), dpi=120)
plt.show()
print('Saved figure2_lada.png and figure3_selector.png to', RUN_BASE)